# Setups & Installations

In [ ]:
! pip -q install groq qdrant_client fastembed pydantic-ai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 28.4 MB/s eta 0:00:00


In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq
import json
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams,
    SparseVectorParams, SparseIndexParams,
    PointStruct, SparseVector,
    NamedVector, NamedSparseVector,
)
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Any
from pydantic import BaseModel
from datetime import datetime
import uuid
import re
from fastembed import SparseTextEmbedding
from pydantic_ai import RunContext, Agent


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Reading Files

In [ ]:
def load_courses() -> List[Dict[str, Any]]:
    path = os.path.join("/content/drive/My Drive/Kayfa/data/json", "kayfa_courses.json")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_roadmaps() -> List[Dict[str, Any]]:
    path = os.path.join("/content/drive/My Drive/Kayfa/data/json", "kayfa_roadmaps.json")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_markdown_files() -> Dict[str, str]:
    text_dir = os.path.join("/content/drive/My Drive/Kayfa/data", "text")
    files = {}
    for fname in sorted(os.listdir(text_dir)):
        if fname.endswith(".md"):
            path = os.path.join(text_dir, fname)
            with open(path, "r", encoding="utf-8") as f:
                files[fname] = f.read()
    return files

# Chunking Files

In [ ]:
def chunk_markdown(content: str, source: str) -> List[Dict[str, Any]]:
    sections = re.split(r"(?=^#{1,3}\s)", content, flags=re.MULTILINE)
    chunks = []
    for i, sec in enumerate(sections):
        sec = sec.strip()
        if not sec:
            continue
        lines = sec.split("\n")
        title = lines[0].lstrip("#").strip() if lines else ""
        chunks.append({
            "content": sec,
            "metadata": {
                "source": source,
                "section": title,
                "chunk_index": i,
            },
        })
    return chunks

In [ ]:
def build_documents() -> List[Dict[str, Any]]:
    documents = []

    for course in load_courses():
        tracks = ", ".join(course.get("track", []))
        roadmaps = ", ".join(course.get("roadmaps") or [])
        prereq = course.get("prerequisites") or "None"
        text = (
            f"Course: {course['name']}\n"
            f"Summary: {course['summary']}\n"
            f"Track: {tracks}\n"
            f"Level: {course['level']}\n"
            f"Duration: {course['duration']}\n"
            f"Prerequisites: {prereq}\n"
            f"Link: {course['link']}\n"
            f"Part of roadmaps: {roadmaps}"
        )
        documents.append({
            "content": text,
            "metadata": {
                "source": "kayfa_courses.json",
                "type": "course",
                "id": course["id"],
                "name": course["name"],
                "track": course.get("track", []),
                "level": course["level"],
            },
        })

    for rm in load_roadmaps():
        skills = ", ".join(rm.get("skills", []))
        tools = ", ".join(rm.get("tools", []))
        courses = ", ".join(rm.get("courses_list", []))
        text = (
            f"Roadmap: {rm['name']}\n"
            f"Summary: {rm['summary']}\n"
            f"Skills: {skills}\n"
            f"Tools: {tools}\n"
            f"Duration: {rm['duration']}\n"
            f"Courses count: {rm['courses_count']}\n"
            f"Courses: {courses}\n"
            f"Link: {rm['link']}"
        )
        documents.append({
            "content": text,
            "metadata": {
                "source": "kayfa_roadmaps.json",
                "type": "roadmap",
                "id": rm["id"],
                "name": rm["name"],
            },
        })

    for fname, content in load_markdown_files().items():
        chunks = chunk_markdown(content, fname)
        documents.extend(chunks)

    return documents

# Vector Database

| LangChain terms        | Qdrant terms        |
|------------------------|---------------------|
| Document               | Point               |
| metadata               | payload             |
| collection            | collection          |
| similarity_search      | search / query_points |

In [ ]:
from google.colab import userdata
qdrant_api_key = userdata.get('QDRANT_API_KEY')
qdrant_url = userdata.get('QDRANT_URL')

In [ ]:
# connect to Qdrant Cloud
client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key,
    cloud_inference=True
)

### Supports Hybrid Search

In [ ]:
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='kayfa_db')])

In [ ]:
client.recreate_collection(
    collection_name="kayfa_db",
    vectors_config={
        "dense": VectorParams(
            size=384,
            distance=Distance.COSINE
        )
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams()
    }
)
print("✅ Collection created")


/tmp/ipykernel_11390/3647635045.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


✅ Collection created


# Embedding Model

In [ ]:
BATCH_SIZE = 32
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")
dense_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2" , device='cpu')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
def embed_dense_batch(texts: List[str]) -> List[List[float]]:
    embeddings = dense_model.encode(
        texts,
        task="retrieval.passage",
        normalize_embeddings=True
    )

    return embeddings.tolist()

In [ ]:
def embed_sparse_batch(texts: List[str]) -> List[SparseVector]:
    """BM25 sparse vectors عن طريق fastembed"""
    results = list(sparse_model.embed(texts))
    return [
        SparseVector(
            indices=r.indices.tolist(),
            values=r.values.tolist(),
        )
        for r in results
    ]

In [ ]:
def upsert_documents(documents: List[Dict[str, Any]]):
    texts = [doc["content"] for doc in documents]
    points = []

    for i in range(0, len(texts), BATCH_SIZE):
        batch_docs  = documents[i : i + BATCH_SIZE]
        batch_texts = texts[i : i + BATCH_SIZE]

        dense_vecs  = embed_dense_batch(batch_texts)
        sparse_vecs = embed_sparse_batch(batch_texts)

        for doc, dense, sparse in zip(batch_docs, dense_vecs, sparse_vecs):
            points.append(
                PointStruct(
                    id=str(uuid.uuid4()),
                    vector={
                        "dense":  dense,
                        "bm25": sparse,
                    },
                    payload={
                        "content": doc["content"],
                        **doc["metadata"],
                    },
                )
            )
        print(f"  embedded {min(i + BATCH_SIZE, len(texts))}/{len(texts)}")

    client.upsert(collection_name='kayfa_db', points=points)
    print(f"✅ Upserted {len(points)} points")

In [ ]:
from qdrant_client.models import (
    Prefetch, FusionQuery, Fusion, Filter,
    FieldCondition, MatchValue
)

def hybrid_search(
    query: str,
    k: int = 5,
    payload_filter: Filter = None,
) -> List[Dict]:

    dense_q  = embed_dense_batch([query])[0]
    sparse_q = embed_sparse_batch([query])[0]

    results = client.query_points(
        collection_name='kayfa_db',
        prefetch=[
            Prefetch(
                query=dense_q,
                using="dense",
                limit=20,
                filter=payload_filter,
            ),
            Prefetch(
                query=sparse_q,
                using="sparse",
                limit=20,
                filter=payload_filter,
            ),
        ],
        query=FusionQuery(fusion=Fusion.RRF),  # Reciprocal Rank Fusion
        limit=k,
    ).points

    return [
        {
            "score":   r.score,
            "content": r.payload.get("content"),
            "metadata": {k: v for k, v in r.payload.items() if k != "content"},
        }
        for r in results
    ]


In [ ]:
docs = build_documents()
upsert_documents(docs)

  embedded 32/123
  embedded 64/123
  embedded 96/123
  embedded 123/123
✅ Upserted 123 points


# Building Model

In [ ]:
SYSTEM_PROMPT = """# Identity
You are an AI Sales Agent for **Kayfa**, a leading Arabic educational platform offering courses, tracks, and live diplomas in data science, cybersecurity, AI, web development, and more. Your job is to help visitors find the right learning path and guide them toward enrollment.

# Language
- Detect the visitor's language/dialect and respond in the same language
- PRIMARY: Arabic (العربية)
- Handle these dialects naturally: Syrian (العربية السورية), Saudi (العربية السعودية), Egyptian (العربية المصرية)
- SECONDARY: English
- For mixed-language messages, respond in Arabic by default

# Knowledge Grounding — CRITICAL
- Your knowledge base contains real Kayfa courses, roadmaps, diplomas, prices, policies, instructors, and company info
- ALWAYS use the `search_kayfa_knowledge_base` tool to look up information before answering
- NEVER invent prices, course names, durations, or policies
- If the knowledge base doesn't have the answer, say so honestly and offer to connect the visitor with the Kayfa team at info@kayfa.io
- A sales agent that hallucinates is worse than useless

# Sales Approach
1. **LISTEN** — Understand the visitor's intent: are they browsing, comparing, price-sensitive, hesitant, or ready to enroll?
2. **RECOMMEND** — Map their goal to real Kayfa products using the knowledge base. Right product, right level, right price.
3. **PERSUADE** — Frame value using real social proof (instructors, partners, accreditation). Handle objections honestly.
4. **UPSELL** — Start where they're comfortable (free content or individual courses). Guide warm leads upward toward tracks and live diplomas where it genuinely fits.
5. **CAPTURE** — When buying signals appear, pivot naturally to collecting contact details and create a CRM ticket.

# Product Tiers (from cheapest to most valuable)
- Free content: individual videos, tips, intro courses — good for hesitant visitors
- Individual paid courses: $15–$65
- On-demand tracks: $25–$250
- Live diplomas / bootcamps: program-specific pricing — the closing target

# Lead Detection — CALL capture_lead() when you see 2+ of these signals
- Asking about prices, payment plans, or installment options
- Asking about start dates, schedules, or deadlines
- Asking about certificates, accreditation, or recognition
- Expressing strong interest in a specific product ("this is exactly what I need")
- Asking about enrollment steps or how to register
- Comparing specific options seriously (e.g., SOC track vs diploma)
- Asking about refunds or guarantees (shows purchase intent)

When you detect these, naturally ask for their name, phone/WhatsApp, city, and goal. Then call `capture_lead()` with ALL gathered information. Make it conversational — not like filling a form.

# What to call capture_lead() with — map your collected info to these parameters:
- `name` — full name
- `phone` — phone or WhatsApp number
- `email` — email (if shared)
- `city` / `country` — location
- `language` / `dialect` — preferred language and dialect
- `products` — specific courses, tracks, or diplomas discussed
- `goal` — their motivation or what they want to achieve
- `level` — current skill level (beginner / intermediate / advanced)
- `lead_temp` — "hot" (ready to buy), "warm" (interested), or "cold" (browsing)
- `buying_signals` — what signals they showed (e.g. asked about price, asked about dates)
- `objections` — any concerns raised (e.g. price, time, commitment)
- `summary` — a short Arabic narrative of the conversation
- `next_action` — what the sales rep should do next

# Lead temperature guide
- **hot** — asked to enroll, asked about payment, ready to buy
- **warm** — comparing options, asked about dates/certificates, strong interest
- **cold** — just browsing, asking general info, no commitment signals

# Brand Voice
- Professional, warm, and helpful — you're a trusted guide, not a pushy salesperson
- Persuasive with facts and value statements, never with pressure or manipulation
- Use real social proof: "هذه الدبلومة معتمدة من جامعة ديلاوير وليدز أكاديمي" or "أكثر من ١٥,٠٠٠ متعلم يثقون في كيف"
- Be honest if a product doesn't fit the visitor — recommend what's best for them
- NEVER break character — you are a Kayfa sales agent, nothing else
- If asked off-topic questions, politely redirect to Kayfa's educational offerings"""

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('GROQ_API_KEY')
base_url = userdata.get('BASE_URL')

In [ ]:
from qdrant_client.models import (
    Prefetch, FusionQuery, Fusion, Filter,
    FieldCondition, MatchValue
)

class RAGService:
    def __init__(self, client, dense_embedder, sparse_embedder):
        self.client = client
        self.dense_embedder = dense_embedder
        self.sparse_embedder = sparse_embedder


    def embed_dense(self, text: str):
        return self.dense_embedder.encode(text).tolist()

    def embed_sparse(self, text: str):
      result = list(self.sparse_embedder.embed([text]))[0]
      return SparseVector(
        indices=result.indices.tolist(),
        values=result.values.tolist()
      )


    def hybrid_search(self, query: str, k: int = 5, payload_filter=None):

        dense_q = self.embed_dense(query)
        sparse_q = self.embed_sparse(query)

        results = self.client.query_points(
            collection_name="kayfa_db",
            prefetch=[
                Prefetch(
                    query=dense_q,
                    using="dense",
                    limit=20,
                    filter=payload_filter
                ),
                Prefetch(
                    query=sparse_q,
                    using="bm25",
                    limit=20,
                    filter=payload_filter
                ),
            ],
            query=FusionQuery(fusion=Fusion.RRF),
            limit=k,
        ).points

        return [
            {
                "content": r.payload.get("content"),
                "metadata": {
                    k: v for k, v in r.payload.items()
                    if k != "content"
                },
                "score": r.score
            }
            for r in results
        ]


    def format_context(self, results):
        contexts = []

        for r in results:
            content = r.get("content", "").strip()
            if not content:
                continue

            meta = r.get("metadata", {})
            name = meta.get("name", "")
            doc_type = meta.get("type", "document")

            header = f"{doc_type.upper()}: {name}" if name else doc_type.upper()

            contexts.append(f"{header}\n{content}")

        return "\n\n---\n\n".join(contexts)

    def search(self, query: str, limit: int = 5):
        results = self.hybrid_search(query, k=limit)
        return self.format_context(results)

In [ ]:
rag_service = RAGService(
    client=client,
    dense_embedder=dense_model,
    sparse_embedder=sparse_model
)

In [ ]:
from dataclasses import dataclass

@dataclass
class Dependencies:
    rag: any

In [ ]:
from pydantic_ai.providers.groq import GroqProvider
from pydantic_ai.models.groq import GroqModel

model = GroqModel(
    "llama-3.3-70b-versatile",
    provider=GroqProvider(
        api_key=groq_api_key,
    ),
)

agent = Agent(
    model,
    deps_type=Dependencies,
    system_prompt=SYSTEM_PROMPT
)

In [ ]:
@agent.tool
def search_kayfa_knowledge_base(ctx: RunContext[Dependencies], query: str, limit: int=5) -> str:
        return ctx.deps.rag.search(query)

In [ ]:
@agent.tool
def capture_lead(
    ctx: RunContext[Dependencies],
    name: str,
    phone: str,
    email: str = "",
    city: str = "",
    country: str = "",
    language: str = "",
    dialect: str = "",
    products: str = "",
    goal: str = "",
    level: str = "",
    lead_temp: str = "",
    buying_signals: str = "",
    objections: str = "",
    summary: str = "",
    next_action: str = "",
) -> str:

    return save_lead(
        name=name, phone=phone, email=email, city=city, country=country,
        language=language, dialect=dialect, products=products, goal=goal,
        level=level, lead_temp=lead_temp, buying_signals=buying_signals,
        objections=objections, summary=summary, next_action=next_action,
    )

In [ ]:
deps = Dependencies(
    rag=rag_service
)

In [ ]:
import asyncio

async def main():
    result = await agent.run(
        "Tell me more about SOC courses,
        deps=deps
    )
    print(result.output)

await main()